# BLM-SGAN: training on Colab

BLM-SGAN conditions SSA-GAN's generator on a BERT text encoder trained for bird captions. This notebook trains it
in two steps and exports the result for the web app:

| Section | What | Time |
|---|---|---|
| 2 | BERT-DAMSM: the BERT text encoder and a DAMSM image encoder, trained together on CUB | ~2 h on a T4 (40 epochs) |
| 3 | The GAN, 600 epochs; FID / IS / R-precision every `EVAL_EVERY` epochs | many sessions; the first log lines print the speed |
| 4 | Export the best generator + its text encoder for `checkpoints/birds/` | seconds |

The reported model was trained with these settings (batch 24 on an A100, no Fourier loss) and is epoch 555.

**Before the first run:** *Runtime → Change runtime type → A100* (or a T4 with `BATCH_SIZE = 12`). Nothing to prepare
on Google Drive: the notebook creates **My Drive/BLM-SGAN/**, takes the CUB captions from the repository and the
images from Caltech. A run needs about 15 GB of free Drive space.

**Sessions:** Colab ends sessions without warning. Both steps write their full state to Drive after every epoch, so
*Runtime → Run all* continues where they stopped.

In [ ]:
import os, sys, glob, shutil, hashlib, subprocess, time, csv

# --- this run --------------------------------------------------------------------------------------
EXPERIMENT = "blm_sgan"     # folder of this run on Drive
FOURIER_WEIGHT = 0.0        # optional Fourier loss (0 = off, as in the reported model)
FOURIER_MODE = "batch"      # spectrum statistics of the generated batch vs the real batch

# --- Google Drive ------------------------------------------------------------------------------------
ROOT = "/content/drive/MyDrive/BLM-SGAN"
DAMSM_DIR = f"{ROOT}/damsm"             # step 2: the BERT-DAMSM encoders (trained once, shared by all runs)
OUT = f"{ROOT}/runs/{EXPERIMENT}"       # step 3: state.pth, samples/, metrics.csv, models/
IMAGE_ENCODER_INIT = f"{ROOT}/image_encoder200.pth"   # optional: AttnGAN's DAMSM image encoder as the starting
                                                      # point of step 2 (otherwise ImageNet weights)

# --- code --------------------------------------------------------------------------------------------
REPO_URL = "https://github.com/MSA-GenAi/BLM-SGAN.git"
REPO_BRANCH = "main"

# --- training ----------------------------------------------------------------------------------------
DAMSM_EPOCHS = 40
GAN_EPOCHS = 600
BATCH_SIZE = 24         # needs ~22 GB (A100); 12 fits a 15-16 GB T4
SAVE_EVERY = 5          # keep models/netG_<epoch>.pth every N epochs (118 MB each), for the epoch selection
EVAL_EVERY = 5          # FID / IS / R-precision on the 2,933 test images -> metrics.csv, models/netG_best.pth
TIME_LIMIT_HOURS = 0    # 0 = until Colab ends the session (the state is saved every epoch)

DATA = "/content/data/birds"   # local disk (fast), rebuilt every session
CODE = "/content/code"
NB_START = time.time()

def hours_left():
    return TIME_LIMIT_HOURS - (time.time() - NB_START) / 3600

def damsm_epochs_done():
    try:
        rows = list(csv.reader(open(f"{DAMSM_DIR}/damsm_log.csv")))[1:]
        return max(int(r[0]) for r in rows if r)
    except (OSError, ValueError):
        return 0

def encoder_ready():   # the GAN must use the finished encoder, never a half-trained one
    return (damsm_epochs_done() >= DAMSM_EPOCHS and os.path.isfile(f"{DAMSM_DIR}/text_encoder.pth")
            and os.path.isfile(f"{DAMSM_DIR}/image_encoder.pth"))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU, then run again"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
if 2.3 + 0.8 * BATCH_SIZE > 0.95 * GPU_GB:   # measured: ~2.3 GB + ~0.8 GB per image
    print(f"WARNING: batch {BATCH_SIZE} probably does not fit in {GPU_GB:.1f} GB")
os.makedirs(DAMSM_DIR, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print(f"run {EXPERIMENT}: batch {BATCH_SIZE}, {GAN_EPOCHS} epochs, Fourier weight {FOURIER_WEIGHT}")
print("run folder:", OUT)
print("text encoder:", "ready" if encoder_ready() else f"{damsm_epochs_done()}/{DAMSM_EPOCHS} epochs - section 2 trains it")

## 1. Code and data

In [ ]:
def github_token():   # only for a private copy: the key icon in the left bar, name GITHUB_TOKEN, "Notebook access" on
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        return ""

if os.path.isdir(CODE):
    shutil.rmtree(CODE)
tok = github_token()
url = REPO_URL.replace("https://", f"https://x-access-token:{tok}@") if tok else REPO_URL
r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", REPO_BRANCH, url, CODE], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError("git clone failed:\n" + (r.stderr.replace(tok, "***") if tok else r.stderr))
subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", REPO_URL])   # never keep a token on disk
commit = subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print("code ready:", CODE, "|", commit)

In [ ]:
CUB_URL = "https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz?download=1"
CUB_MD5 = "97eceeb196236b17998738112f37df78"

def md5(path, chunk=1 << 22):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

if not os.path.isdir(f"{DATA}/CUB_200_2011/images"):
    os.makedirs(DATA, exist_ok=True)
    !wget -q -O /tmp/CUB_200_2011.tgz "{CUB_URL}"
    assert md5("/tmp/CUB_200_2011.tgz") == CUB_MD5, "CUB archive is corrupted - delete it and rerun"
    !tar -xzf /tmp/CUB_200_2011.tgz -C {DATA} CUB_200_2011/images CUB_200_2011/images.txt CUB_200_2011/bounding_boxes.txt
    os.remove("/tmp/CUB_200_2011.tgz")

META = f"{CODE}/training/cub_meta/birds"   # captions + train/test splits (AttnGAN's preprocessed CUB metadata)
for f in ["captions.pickle", "train/filenames.pickle", "train/class_info.pickle",
          "test/filenames.pickle", "test/class_info.pickle"]:
    os.makedirs(os.path.dirname(f"{DATA}/{f}"), exist_ok=True)
    shutil.copy(f"{META}/{f}", f"{DATA}/{f}")
n = len(glob.glob(f"{DATA}/CUB_200_2011/images/*/*.jpg"))
assert n == 11788, n
print(n, "images ready in", DATA)

## 2. BERT-DAMSM text and image encoders (once)

Skipped when `DAMSM_DIR` already holds a finished encoder. Validation prints the R-precision of **real** test images
against their captions (top-1 of 100, chance 1%): the encoder of the reported model reached 16.3%, and AttnGAN's
LSTM-DAMSM scores 16.9% on the same test.

In [ ]:
if encoder_ready():
    print(f"encoder ready ({damsm_epochs_done()} epochs) - nothing to train")
else:
    use_init = bool(IMAGE_ENCODER_INIT) and os.path.isfile(IMAGE_ENCODER_INIT)
    print("image encoder start:", IMAGE_ENCODER_INIT if use_init else "ImageNet weights")
    init = f"--init_image_encoder '{IMAGE_ENCODER_INIT}'" if use_init else ""
    limit = f"{hours_left():.2f}" if TIME_LIMIT_HOURS else "0"
    !cd {CODE} && python -u training/train_damsm.py --data_dir {DATA} --out_dir "{DAMSM_DIR}" --epochs {DAMSM_EPOCHS} --time_limit_hours {limit} {init}
    if globals().get("_exit_code", 0) != 0:   # set by IPython after a ! command
        raise RuntimeError("train_damsm.py failed - see the messages above")
!tail -n 3 "{DAMSM_DIR}/damsm_log.csv"

## 3. GAN training

Every epoch the full state goes to `state.pth` in the run folder; every 5 epochs a grid `samples/epoch_XXX.png`
(8 test captions × 4 noise vectors, captions in `samples/captions.txt`). Every `EVAL_EVERY` epochs: FID, IS and
R-precision in `metrics.csv`; the best-FID generator is `models/netG_best.pth`. The first epoch's step lines print the
speed (s/step, estimated epoch time) and the peak GPU memory.

In [ ]:
if not encoder_ready():
    raise RuntimeError(f"the text encoder is not finished ({damsm_epochs_done()}/{DAMSM_EPOCHS} epochs) - run section 2")
flags = (f"--fourier_weight {FOURIER_WEIGHT} --fourier_mode {FOURIER_MODE} --save_every {SAVE_EVERY} "
         f"--eval_every {EVAL_EVERY}")
limit = f"{hours_left():.2f}" if TIME_LIMIT_HOURS else "0"
!cd {CODE} && python -u training/train_gan.py --data_dir {DATA} --damsm_dir "{DAMSM_DIR}" --out_dir "{OUT}" --epochs {GAN_EPOCHS} --batch_size {BATCH_SIZE} --gpus 1 --time_limit_hours {limit} {flags}
if globals().get("_exit_code", 0) != 0:   # everything up to the last finished epoch is on Drive
    raise RuntimeError("train_gan.py failed - see the messages above")

In [ ]:
# progress (also works on its own after a restart: run the first cell, then this one)
import pandas as pd
from IPython.display import Image, display
def table(name):
    try:
        return pd.read_csv(f"{OUT}/{name}")
    except (OSError, pd.errors.EmptyDataError):
        return pd.DataFrame()
log, met = table("train_log.csv"), table("metrics.csv")
print(log.tail(3).to_string(index=False) if len(log) else "no finished GAN epoch yet")
if len(met):
    print(met.to_string(index=False))
    if len(met) > 1:
        met.plot(x="epoch", y=["fid", "r_precision"], subplots=True, figsize=(8, 5), marker="o")
samples = sorted(glob.glob(f"{OUT}/samples/epoch_*.png"))
if samples:
    print(open(f"{OUT}/samples/captions.txt").read()); display(Image(samples[-1]))

## 4. Export for the web app

Writes one zip with the best generator (by FID during training) and its text encoder to the run folder on Drive.
Unzip it into `checkpoints/birds/` of the repository and restart the app. To choose the epoch more carefully, score
every saved generator with `colab_eval_epochs.ipynb` and export that epoch with `training/export_model.py`.

In [ ]:
import json, re
best = f"{OUT}/models/best.txt"
if not os.path.isfile(best):
    print("nothing to export yet: the first evaluation is at epoch", EVAL_EVERY)
else:
    epoch = int(re.search(r"epoch (\d+)", open(best).read()).group(1))
    exp = "/content/export/blm_sgan"
    os.makedirs(exp, exist_ok=True)
    shutil.copy(f"{OUT}/models/netG_best.pth", f"{exp}/netG_{epoch:03d}.pth")
    shutil.copy(f"{OUT}/models/text_encoder.pth", f"{exp}/text_encoder.pth")
    json.dump({"name": "BLM-SGAN", "kind": "blm", "order": 5,
               "description": "Our model: a BERT text encoder trained on bird captions guides an SSA-GAN generator."},
              open(f"{exp}/model.json", "w"), indent=1)
    zip_path = shutil.make_archive(f"{OUT}/blm_sgan_epoch{epoch}", "zip", "/content/export", "blm_sgan")
    print("zip on Drive:", zip_path)